In [ ]:
from google.colab import drive
import polars as pl  # Using Polars for multi-threaded, memory-efficient data processing as compared to pandas dataframe for large datasets
import os
!pip install rapidfuzz
from rapidfuzz.distance import JaroWinkler

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define the exact path where your files are uploaded
data_dir = '/content/drive/MyDrive/ml-challenge'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

# 1. Load ground truth and explode it fully (ground truth is usually smaller)
ground_truth = pl.read_csv(f"{data_dir}/train_ground_truth.tsv", separator="\t")
exploded_gt = (
    ground_truth
    .filter(pl.col("matched_entity_ids").is_not_null())
    .with_columns(candidate_entity_id=pl.col("matched_entity_ids").str.split(","))
    .explode("candidate_entity_id")
    .select([
        pl.col("source1_entity_id"),
        pl.col("candidate_entity_id").str.strip_chars()
    ])
    .unique()
)

# Convert exploded_gt to a Python set of tuples for extremely fast O(1) membership checking in Python
# (This avoids heavy memory joins on 2M+ rows)
print("Building ground truth lookup set...")
gt_set = set(
    exploded_gt.select(["source1_entity_id", "candidate_entity_id"])
    .iter_rows()
)
print(f"Total true match pairs loaded: {len(gt_set)}")

# 2. Process candidate_pairs in chunks or stream line-by-line / batch-by-batch
# Alternatively, read csv using Polars batch scanning if file is large,
# or process dataframe using lazy evaluation or smaller slice loops.

Building ground truth lookup set...
Total true match pairs loaded: 7638365


In [ ]:
# View the first 10 items by converting a slice of the set to a list
print(list(gt_set)[:19])

[('S1-368518377', 'S3-507811233'), ('S1-408717665', 'S3-615067486'), ('S1-590292422', 'S3-754485431'), ('S1-861363476', 'S2-627119842'), ('S1-816631772', 'S3-351106150'), ('S1-404202193', 'S3-992769866'), ('S1-524882924', 'S3-821227839'), ('S1-488939686', 'S2-338875620'), ('S1-562277743', 'S2-457019960'), ('S1-667502531', 'S2-572359378'), ('S1-322538414', 'S3-259724442'), ('S1-327745331', 'S2-560019844'), ('S1-637507029', 'S3-564250815'), ('S1-783538994', 'S2-215109098'), ('S1-625581522', 'S3-617218902'), ('S1-362741249', 'S2-231116815'), ('S1-258844971', 'S3-602904635'), ('S1-246674001', 'S2-576996664'), ('S1-120314249', 'S2-521812706')]


In [ ]:


output_file = f"{data_dir}/labeled_train_pairs.tsv"

# Remove existing output file if it exists
if os.path.exists(output_file):
    os.remove(output_file)

chunk_size = 100000  # Process 100k rows at a time to prevent RAM crashes
is_first_chunk = True

print("Processing candidate pairs in chunks...")
candidate_df = pl.read_csv(f"{data_dir}/candidate_pairs.tsv", separator="\t")
total_rows = candidate_df.height

for start_idx in range(0, total_rows, chunk_size):
    end_idx = min(start_idx + chunk_size, total_rows)
    chunk = candidate_df.slice(start_idx, end_idx - start_idx)

    # Explode the comma-separated candidate entity IDs into individual rows
    exploded_chunk = (
        chunk
        .filter(pl.col("candidate_entity_ids").is_not_null())
        .with_columns(
            candidate_entity_id=pl.col("candidate_entity_ids").str.split(",")
        )
        .explode("candidate_entity_id")
        .select([
            pl.col("source1_entity_id"),
            pl.col("candidate_entity_id").str.strip_chars()
        ])
        .unique()
    )

    # Check each pair against your ground truth set (1 for match, 0 for non-match)
    pairs_list = exploded_chunk.select(["source1_entity_id", "candidate_entity_id"]).iter_rows()
    labels = [1 if (s1, s2) in gt_set else 0 for s1, s2 in pairs_list]

    labeled_chunk = exploded_chunk.with_columns(label=pl.Series(labels))

    # Append the labeled chunk directly to disk
    labeled_chunk.write_csv(
        output_file,
        separator="\t",
        include_header=is_first_chunk,
        quote_style="never"
    )

    is_first_chunk = False
    print(f"Processed rows {start_idx} to {end_idx}...")

print("Finished labeling and saving all candidate pairs successfully!")

Processing candidate pairs in chunks...
Processed rows 0 to 100000...
Processed rows 100000 to 200000...
Processed rows 200000 to 300000...
Processed rows 300000 to 400000...
Processed rows 400000 to 500000...
Processed rows 500000 to 600000...
Processed rows 600000 to 700000...
Processed rows 700000 to 800000...
Processed rows 800000 to 900000...
Processed rows 900000 to 1000000...
Processed rows 1000000 to 1100000...
Processed rows 1100000 to 1200000...
Processed rows 1200000 to 1300000...
Processed rows 1300000 to 1400000...
Processed rows 1400000 to 1500000...
Processed rows 1500000 to 1600000...
Processed rows 1600000 to 1700000...
Processed rows 1700000 to 1800000...
Processed rows 1800000 to 1900000...
Processed rows 1900000 to 2000000...
Processed rows 2000000 to 2100000...
Processed rows 2100000 to 2200000...
Processed rows 2200000 to 2206821...
Finished labeling and saving all candidate pairs successfully!


In [ ]:

# Load the file without assuming a header row, then assign correct column names
labeled_df = pl.read_csv(
    f"{data_dir}/labeled_train_pairs.tsv",
    separator="\t",
    has_header=False
).rename({"column_1": "source1_entity_id", "column_2": "candidate_entity_id", "column_3": "label"})

print(labeled_df.head(20))

# Check class distribution safely now
label_counts = labeled_df.group_by("label").len()
print("\nLabel Distribution:")
print(label_counts)

shape: (20, 3)
┌───────────────────┬─────────────────────┬───────┐
│ source1_entity_id ┆ candidate_entity_id ┆ label │
│ ---               ┆ ---                 ┆ ---   │
│ str               ┆ str                 ┆ i64   │
╞═══════════════════╪═════════════════════╪═══════╡
│ S1-411294365      ┆ S2-785205660        ┆ 0     │
│ S1-256245238      ┆ S3-29917669         ┆ 0     │
│ S1-124262910      ┆ S3-103719086        ┆ 0     │
│ S1-650219917      ┆ S3-138916455        ┆ 0     │
│ S1-937987387      ┆ S2-682145673        ┆ 0     │
│ …                 ┆ …                   ┆ …     │
│ S1-955822706      ┆ S2-917861087        ┆ 0     │
│ S1-753678595      ┆ S2-710807225        ┆ 0     │
│ S1-921230480      ┆ S3-578526573        ┆ 0     │
│ S1-966742860      ┆ S3-545942289        ┆ 0     │
│ S1-693811789      ┆ S2-895503181        ┆ 0     │
└───────────────────┴─────────────────────┴───────┘

Label Distribution:
shape: (2, 2)
┌───────┬────────┐
│ label ┆ len    │
│ ---   ┆ ---    │
│ i64   ┆

In [ ]:


# Load your blocked parquet files
source1_blocked = pl.read_parquet(f"{data_dir}/source1_blocked.parquet")
source2_blocked = pl.read_parquet(f"{data_dir}/source2_blocked.parquet")
source3_blocked = pl.read_parquet(f"{data_dir}/source3_blocked.parquet")


In [ ]:
# Combine source 2 and source 3 blocked tables into a single lookup dataframe for candidates
candidate_sources = pl.concat([source2_blocked, source3_blocked], how="diagonal")

print("Combined candidate sources shape:", candidate_sources.shape)
print(candidate_sources.head(2))

Combined candidate sources shape: (10320219, 9)
shape: (2, 9)
┌────────────┬───────────┬───────────┬─────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ entity_id  ┆ business_ ┆ business_ ┆ country ┆ … ┆ address_p ┆ initials_ ┆ name_len_ ┆ blocking_ │
│ ---        ┆ name      ┆ address   ┆ ---     ┆   ┆ refix     ┆ key       ┆ bin       ┆ key       │
│ str        ┆ ---       ┆ ---       ┆ str     ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆ str       ┆ str       ┆         ┆   ┆ str       ┆ str       ┆ str       ┆ str       │
╞════════════╪═══════════╪═══════════╪═════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ S2-1663764 ┆ ra ma ma  ┆ kh no.    ┆ india   ┆ … ┆ 99999     ┆ dgiklmmmp ┆ L         ┆ india_iva │
│ 19         ┆ ra ka ta  ┆ -570/13,  ┆         ┆   ┆           ┆ rrrttt    ┆           ┆ _dgiklmmm │
│            ┆ ga pa ra  ┆ new       ┆         ┆   ┆           ┆           ┆           ┆ prrrttt_L │
│            ┆ iva…      ┆ de

In [ ]:
print("Merging Source 1 attributes...")
merged = labeled_df.join(
    source1_blocked.select(["entity_id", "business_name", "business_address", "country"]),
    left_on="source1_entity_id",
    right_on="entity_id",
    how="inner"
).rename({
    "business_name": "s1_name",
    "business_address": "s1_address",
    "country": "s1_country"
})

print("Merging Candidate attributes...")
merged = merged.join(
    candidate_sources.select(["entity_id", "business_name", "business_address", "country"]),
    left_on="candidate_entity_id",
    right_on="entity_id",
    how="inner"
).rename({
    "business_name": "cand_name",
    "business_address": "cand_address",
    "country": "cand_country"
})

print(f"Successfully merged! Total rows ready for feature extraction: {merged.height}")

Merging Source 1 attributes...
Merging Candidate attributes...
Successfully merged! Total rows ready for feature extraction: 126109


In [ ]:
print(merged.shape)
print(merged.head(5))

(126109, 9)
shape: (5, 9)
┌────────────┬────────────┬───────┬────────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ source1_en ┆ candidate_ ┆ label ┆ s1_name    ┆ … ┆ s1_countr ┆ cand_name ┆ cand_addr ┆ cand_coun │
│ tity_id    ┆ entity_id  ┆ ---   ┆ ---        ┆   ┆ y         ┆ ---       ┆ ess       ┆ try       │
│ ---        ┆ ---        ┆ i64   ┆ str        ┆   ┆ ---       ┆ str       ┆ ---       ┆ ---       │
│ str        ┆ str        ┆       ┆            ┆   ┆ str       ┆           ┆ str       ┆ str       │
╞════════════╪════════════╪═══════╪════════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ S1-8544786 ┆ S2-4410084 ┆ 0     ┆ sree       ┆ … ┆ india     ┆ sree      ┆ #77-1-2   ┆ india     │
│ 90         ┆ 4          ┆       ┆ vidyalaya  ┆   ┆           ┆ ventures  ┆ gandhipur ┆           │
│            ┆            ┆       ┆            ┆   ┆           ┆           ┆ am-3      ┆           │
│            ┆            ┆       ┆            ┆   ┆           ┆ 

In [ ]:
from rapidfuzz.distance import JaroWinkler

# 1. Compute Name Similarity Scores
name_scores = [
    JaroWinkler.normalized_similarity(str(n1), str(n2)) if n1 is not None and n2 is not None else 0.0
    for n1, n2 in zip(merged["s1_name"].to_list(), merged["cand_name"].to_list())
]

# 2. Compute Address Similarity Scores (Assuming your address columns are named 's1_address' and 'cand_address')
# Note: Adjust column names if your address columns have a different prefix (e.g., 's1_addr', 'cand_addr')
addr_scores = [
    JaroWinkler.normalized_similarity(str(a1), str(a2)) if a1 is not None and a2 is not None else 0.0
    for a1, a2 in zip(merged["s1_address"].to_list(), merged["cand_address"].to_list())
]

# 3. Add both features to your Polars DataFrame
merged = merged.with_columns([
    pl.Series("jw_name_score", name_scores),
    pl.Series("jw_addr_score", addr_scores)
])
# Add country match feature (1 if countries match, 0 otherwise)
merged = merged.with_columns(
    country_match=pl.when(pl.col("s1_country") == pl.col("cand_country"))
    .then(1)
    .otherwise(0)
)
# Preview the new features
print(merged.select(['source1_entity_id', 'candidate_entity_id',"s1_address","cand_address","s1_name", "cand_name", "jw_name_score", "jw_addr_score","country_match","label"]).head(5))


shape: (5, 10)
┌────────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────┐
│ source1_en ┆ candidate_ ┆ s1_address ┆ cand_addr ┆ … ┆ jw_name_s ┆ jw_addr_s ┆ country_m ┆ label │
│ tity_id    ┆ entity_id  ┆ ---        ┆ ess       ┆   ┆ core      ┆ core      ┆ atch      ┆ ---   │
│ ---        ┆ ---        ┆ str        ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ i64   │
│ str        ┆ str        ┆            ┆ str       ┆   ┆ f64       ┆ f64       ┆ i32       ┆       │
╞════════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════╡
│ S1-8544786 ┆ S2-4410084 ┆ plot no.1  ┆ #77-1-2   ┆ … ┆ 0.630037  ┆ 0.646276  ┆ 1         ┆ 0     │
│ 90         ┆ 4          ┆ bramaramba ┆ gandhipur ┆   ┆           ┆           ┆           ┆       │
│            ┆            ┆ ,          ┆ am-3      ┆   ┆           ┆           ┆           ┆       │
│            ┆            ┆ mallikar…  ┆ near, pr… ┆   ┆           ┆        

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# 1. Select feature columns and target label
feature_cols = ["jw_name_score", "jw_addr_score", "country_match"]
target_col = "label"

# Convert to pandas for easy integration with XGBoost / scikit-learn
df_pd = merged.select(feature_cols + [target_col]).to_pandas()

X = df_pd[feature_cols]
y = df_pd[target_col]

# 2. Split into training and validation sets (80% train, 20% val)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Initialize and train the XGBoost Classifier
model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    eval_metric="logloss"
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=True
)

# 4. Evaluate performance on validation set
preds = model.predict(X_val)
probs = model.predict_proba(X_val)[:, 1]
import joblib
from google.colab import files

# 1. Save the model to your environment or Google Drive folder
model_filename = f"{data_dir}/xgboost_entity_matcher.pkl"
joblib.dump(model, model_filename)
print(f"Model saved successfully to {model_filename}")

# 2. Download the model file directly to your local computer
files.download(model_filename)

print("\nValidation ROC-AUC Score:", roc_auc_score(y_val, probs))
print("\nClassification Report:")
print(classification_report(y_val, preds))
print(f"Positive class ratio in train: {y_train.mean():.4f}")

[0]	validation_0-logloss:0.14258
[1]	validation_0-logloss:0.12444
[2]	validation_0-logloss:0.11137
[3]	validation_0-logloss:0.10115
[4]	validation_0-logloss:0.09285
[5]	validation_0-logloss:0.08592
[6]	validation_0-logloss:0.08006
[7]	validation_0-logloss:0.07503
[8]	validation_0-logloss:0.07070
[9]	validation_0-logloss:0.06694
[10]	validation_0-logloss:0.06365
[11]	validation_0-logloss:0.06078
[12]	validation_0-logloss:0.05824
[13]	validation_0-logloss:0.05602
[14]	validation_0-logloss:0.05404
[15]	validation_0-logloss:0.05230
[16]	validation_0-logloss:0.05077
[17]	validation_0-logloss:0.04940
[18]	validation_0-logloss:0.04822
[19]	validation_0-logloss:0.04716
[20]	validation_0-logloss:0.04624
[21]	validation_0-logloss:0.04540
[22]	validation_0-logloss:0.04467
[23]	validation_0-logloss:0.04401
[24]	validation_0-logloss:0.04345
[25]	validation_0-logloss:0.04296
[26]	validation_0-logloss:0.04253
[27]	validation_0-logloss:0.04213
[28]	validation_0-logloss:0.04179
[29]	validation_0-loglos

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Validation ROC-AUC Score: 0.9909913779877881

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     23805
           1       0.94      0.83      0.88      1417

    accuracy                           0.99     25222
   macro avg       0.97      0.91      0.94     25222
weighted avg       0.99      0.99      0.99     25222

Positive class ratio in train: 0.0562


In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_curve, classification_report

print("Finding optimal threshold for F_0.5 score...")
y_proba = model.predict_proba(X_val)[:, 1]

# Calculate precision, recall for various thresholds
precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)

# Calculate F_0.5 score for each threshold using valid variable names
beta_sq = 0.5 ** 2
f_half_scores = (1 + beta_sq) * (precisions * recalls) / ((beta_sq * precisions) + recalls + 1e-10)

# Find the threshold that maximizes F_0.5
best_idx = np.argmax(f_half_scores)
best_threshold = thresholds[best_idx]
best_f_half = f_half_scores[best_idx]

print(f"Optimal Decision Threshold: {best_threshold:.4f}")
print(f"Best Validation F_0.5 Score: {best_f_half:.4f}")

# Evaluate with the new threshold
y_pred_tuned = (y_proba >= best_threshold).astype(int)
print("\nClassification Report with Tuned Threshold:")
print(classification_report(y_val, y_pred_tuned))

Finding optimal threshold for F_0.5 score...
Optimal Decision Threshold: 0.7853
Best Validation F_0.5 Score: 0.9247

Classification Report with Tuned Threshold:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     23805
           1       0.96      0.80      0.87      1417

    accuracy                           0.99     25222
   macro avg       0.98      0.90      0.93     25222
weighted avg       0.99      0.99      0.99     25222

